# Adding Integer Variables

In the last lesson, we introduced integer variables to our linear optimization models. To get a better intuition for the solution space of integer optimization models, let's revisit the washer and dryer optimization example from last module. In this model, we were maximizing profit based on the number of washers and dryers produced perday, subject to manufacturing, assembly and construction constraints.

When we first looked at this model, we did not require the decision variables (number of washers and dryers produced) to take integer values. This can be a decent approximation, especially if considering averages over time, but it might be more realistic to allow these decisions to take on only integer values. In cases where the decision variable is encoding a binary decision (yes or no), the integer interpretation is often especially important.

Looking at the interactive plot below (make sure to run the cell!), we see that our variables are restricted to integers (the grey dots within what was originally the filled in solution space when we allowed any nonnegative real values). We now can see that our optimal solutions are no longer guarenteed to be found at corners of the solution space: indeed, the original corner that was optimal in the linear optimization model does not fall on all integer values of the decision variables, so is no longer a feasible solution. This makes integer optimization models much harder to solve - as we will see in a later lesson. Try adjusting the sliders again in the interactive plot. What values of the decision variables give you the highest objective function value while satisfying the constraints?

In [ ]:
#@title Integer Model Solution Space

# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Math

# Define the coefficients for the objective function and constraints
objective_coeffs = [100, 120]  # Coefficients of the objective function
constraint_coeffs = [
    [1, 2],  # Manufacturing constraint
    [2, 1],  # Assembly constraint
    [2, 2]   # Testing constraint
]
constraint_bounds = [20, 20, 25]  # Right-hand side values for constraints

# Define level sets for the objective function
level_values = np.arange(200, 2400, 400)  # Level sets from 200 to 2200 at intervals of 400
level_colors = plt.cm.viridis(np.linspace(0, 1, len(level_values)))  # Generate unique colors for each level set

# Create integer sliders for decision variables
x1_slider = widgets.IntSlider(value=0, min=0, max=10, step=1, description="x₁ (Washers)")
x2_slider = widgets.IntSlider(value=0, min=0, max=15, step=1, description="x₂ (Dryers)")

# Create a checkbox for level sets
level_set_checkbox = widgets.Checkbox(value=False, description="Show Objective Level Sets")

# Output widget for displaying equations
output = widgets.Output()

# Function to compute the feasible region plot
def plot_feasible_region(x1, x2, show_level_sets=False):
    # Generate plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Generate the lines for constraints extending to x and y axes
    x_full = np.linspace(0, 15, 300)  # Extended x range
    y1_full = (constraint_bounds[0] - constraint_coeffs[0][0] * x_full) / constraint_coeffs[0][1]  # Manufacturing
    y2_full = (constraint_bounds[1] - constraint_coeffs[1][0] * x_full) / constraint_coeffs[1][1]  # Assembly
    y3_full = (constraint_bounds[2] - constraint_coeffs[2][0] * x_full) / constraint_coeffs[2][1]  # Testing

    # Plot the constraints as extended lines
    ax.plot(x_full, y1_full, label="Manufacturing (Red)", color='red')
    ax.plot(x_full, y2_full, label="Assembly (Blue)", color='blue')
    ax.plot(x_full, y3_full, label="Testing (Green)", color='green')

    # Loop over integer x1 and x2 values to find feasible points
    feasible_points = []
    for x1_int in range(0, 11):
        for x2_int in range(0, 16):
            # Check if the point satisfies all constraints
            constraint_values = [
                constraint_coeffs[0][0] * x1_int + constraint_coeffs[0][1] * x2_int,  # Manufacturing
                constraint_coeffs[1][0] * x1_int + constraint_coeffs[1][1] * x2_int,  # Assembly
                constraint_coeffs[2][0] * x1_int + constraint_coeffs[2][1] * x2_int   # Testing
            ]
            if all(constraint_values[i] <= constraint_bounds[i] for i in range(len(constraint_bounds))):
                feasible_points.append((x1_int, x2_int))

    # Plot feasible integer points
    for point in feasible_points:
        ax.scatter(*point, color='gray', s=50, label="Feasible Point" if point == feasible_points[0] else None)

    # Plot the current slider point
    ax.scatter(x1, x2, color='black', s=100, label="Current Point")
    ax.text(x1 + 0.2, x2 + 0.2, f"({x1}, {x2})", fontsize=10, color='black', bbox=dict(facecolor='yellow', alpha=0.5))

    # Plot level sets if checkbox is checked
    if show_level_sets:
        for level, color in zip(level_values, level_colors):
            y_level = (level - objective_coeffs[0] * x_full) / objective_coeffs[1]
            ax.plot(x_full, y_level, linestyle='--', color=color, label=f"Z = {level}")

    # Set plot limits and labels
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 15)
    ax.set_xlabel(r"$x_1$ (Washers)")
    ax.set_ylabel(r"$x_2$ (Dryers)")
    ax.legend()
    ax.grid(True)
    plt.show()

# Function to dynamically update the equations and plot
def update(change):
    with output:
        # Clear previous output
        output.clear_output(wait=True)

        # Get current slider values
        x1 = x1_slider.value
        x2 = x2_slider.value

        # Compute the objective function value
        Z = objective_coeffs[0] * x1 + objective_coeffs[1] * x2

        # Compute constraint values
        constraint_values = [
            constraint_coeffs[0][0] * x1 + constraint_coeffs[0][1] * x2,  # Manufacturing
            constraint_coeffs[1][0] * x1 + constraint_coeffs[1][1] * x2,  # Assembly
            constraint_coeffs[2][0] * x1 + constraint_coeffs[2][1] * x2   # Testing
        ]

        # Determine constraint statuses
        statuses = [
            r"\textcolor{green}{Satisfied}" if constraint_values[0] <= constraint_bounds[0] else r"\textcolor{red}{Violated}",
            r"\textcolor{green}{Satisfied}" if constraint_values[1] <= constraint_bounds[1] else r"\textcolor{red}{Violated}",
            r"\textcolor{green}{Satisfied}" if constraint_values[2] <= constraint_bounds[2] else r"\textcolor{red}{Violated}"
        ]

        # Display the objective function
        objective_latex = rf"Maximize \, 100x_1 + 120x_2 = 100({x1}) + 120({x2}) = {Z}"
        display(Math(objective_latex))

        # Display the constraints
        for i, (coeff, bound, value, status) in enumerate(zip(constraint_coeffs, constraint_bounds, constraint_values, statuses)):
            constraint_latex = rf"{coeff[0]}x_1 + {coeff[1]}x_2 \leq {bound}" \
                               rf" \quad \text{{where }} {coeff[0]}({x1}) + {coeff[1]}({x2}) = {value}" \
                               rf" \quad \textbf{{{status}}}"
            display(Math(constraint_latex))

        # Add positive integer constraints
        positive_integer_latex = r"x_1, x_2 \in \mathbb{Z}^+"
        display(Math(positive_integer_latex))

        # Plot the feasible region
        plot_feasible_region(x1, x2, show_level_sets=level_set_checkbox.value)

# Attach the update function to sliders and checkbox
x1_slider.observe(update, names='value')
x2_slider.observe(update, names='value')
level_set_checkbox.observe(update, names='value')

# Display the layout
layout = widgets.VBox([x1_slider, x2_slider, level_set_checkbox, output])
display(layout)

# Initialize the equations and plot
update(None)
